# M11 · Clean NULLs in already-loaded mart rows

**Goal:** convert NULL → 0 on every nullable mart column so the BI dashboard
never renders empty cells. Future loads are already hardened in sql/20, 30,
31, 32 (COALESCE / ELSE 0). This notebook fixes the rows that were loaded
*before* the hardening.

Then it re-creates the BI views (33, 34, 35) which were also patched
(NULL→20 deltas, ELSE 0 ratios, COALESCE on lateral repair-hours).

**Idempotent.** Re-running is a no-op once everything is clean.

In [1]:
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src)); break

from accent_fleet.db import get_engine, run_sql_file, transaction
from sqlalchemy import text
import pandas as pd

## 2. Pre-cleanup audit — how many NULL cells per mart?

In [2]:
AUDIT_SQL = text('''
SELECT 'mart_device_monthly_behavior' AS mart,
       COUNT(*) FILTER (WHERE stddev_trip_distance IS NULL
                          OR overspeed_per_100km   IS NULL
                          OR overspeed_per_trip    IS NULL
                          OR avg_speed_over_limit  IS NULL
                          OR speed_alert_per_100km IS NULL
                          OR in_path_stop_ratio    IS NULL
                          OR stops_per_trip        IS NULL
                          OR avg_working_hours     IS NULL) AS rows_with_null,
       COUNT(*) AS total_rows
FROM marts.mart_device_monthly_behavior
UNION ALL
SELECT 'mart_fleet_daily',
       COUNT(*) FILTER (WHERE total_driving_hours IS NULL OR avg_max_speed_kmh IS NULL),
       COUNT(*) FROM marts.mart_fleet_daily
UNION ALL
SELECT 'mart_vehicle_monthly',
       COUNT(*) FILTER (WHERE total_driving_hours IS NULL
                          OR trip_fuel_used_l    IS NULL
                          OR avg_cost_per_litre  IS NULL
                          OR cost_per_km         IS NULL
                          OR fuel_l_per_100km    IS NULL),
       COUNT(*) FROM marts.mart_vehicle_monthly
UNION ALL
SELECT 'mart_tenant_monthly_summary',
       COUNT(*) FILTER (WHERE total_driving_hours      IS NULL
                          OR avg_distance_per_vehicle IS NULL
                          OR cost_per_km              IS NULL),
       COUNT(*) FROM marts.mart_tenant_monthly_summary
''')
with get_engine().connect() as conn:
    audit_before = pd.read_sql(AUDIT_SQL, conn)
audit_before

,mart,rows_with_null,total_rows
0,mart_tenant_monthly_summary,79,395
1,mart_fleet_daily,2386,11883
2,mart_vehicle_monthly,24497,24497
3,mart_device_monthly_behavior,23225,23965


## 3. Run the cleanup

In [3]:
with transaction() as conn:
    run_sql_file(conn, '29_clean_mart_nulls.sql')
print('cleanup applied.')

cleanup applied.


## 4. Rebuild BI views with hardened ratios

In [4]:
VIEWS = [
    '33_v_executive_dashboard.sql',
    '34_v_operational_dashboard.sql',
    '35_v_maintenance_dashboard.sql',
]
with transaction() as conn:
    for f in VIEWS:
        run_sql_file(conn, f)
        print(f'ok: {f}')

ok: 33_v_executive_dashboard.sql
ok: 34_v_operational_dashboard.sql
ok: 35_v_maintenance_dashboard.sql


## 5. Post-cleanup audit — should be all zeros

In [5]:
with get_engine().connect() as conn:
    audit_after = pd.read_sql(AUDIT_SQL, conn)
audit_after

,mart,rows_with_null,total_rows
0,mart_tenant_monthly_summary,0,395
1,mart_fleet_daily,0,11883
2,mart_vehicle_monthly,0,24497
3,mart_device_monthly_behavior,0,23965
